# Test set scoring

This notebook scores every frozen model on new data, with fixed settings, and saves the results. We use it twice:
1. **Dry run on Dataset 2** (`RUN_ON = "dataset2"`). Every number must match the shared comparison (`evaluation/results/comparison_gap2_grace3.csv`). This shows the code scores each model the same way as before.
2. **One run on the Test set** (`RUN_ON = "test"`), only after the team freezes all choices.

**What is frozen.** Nothing is trained, refitted or recalibrated here. We only load saved models and saved thresholds.
- Ridge: 28 per-sensor models (`models/Ridge/ridge_model.joblib`), calibration in `models/Ridge/calibration/calibration.pt`.
- Transformer: 12-hour window, seed 0 (`models/Transformer/Notebook/Model Weights/best_model.pt`), calibration in `calibration/calibration.pt` next to it.
- XGBoost: Suhas's 5 fold models per seed (`results/xgboost_iforest_ensemble/xgb/seed{k}/models/`), scored in turn by row position.
- IF + PCA: Suhas's Isolation Forest with PCA, seeds 0, 1 and 2 (`models/Ensemble/iforest_pca_models/seed{k}.joblib`).
- Ensemble: the max of the XGBoost and IF + PCA percentiles, seed k with seed k.
- Ridge and the Transformer use the `topk_percentile` score with the top 3 sensors.
- All thresholds were set on the Calibration block (Dataset 1), inside the budget of 1% of hours and 1 false-alarm episode per week.
- Episodes: alarm runs with 2 or fewer quiet hours between them count as one (merge gap 2). The 3 hours after each attack are not counted (grace 3).
- The first 23 hours are not scored, but they are still fed in as history (the 24-hour XGBoost features need them).

**The switch.** `RUN_ON` picks the data. `UNLOCK_TEST` must be set to `True` by hand before `RUN_ON = "test"` works. Otherwise the first cell stops with an error.

**The Test set is scored once only.** After that run, we do not change any model, threshold or setting, and we report what comes out.

Steps:
1. Setup
2. Load the data
3. Ridge
4. Transformer
5. XGBoost, IF + PCA and the ensemble
6. Summary of all models
7. Check against the shared comparison (dry run only)

## 1. Setup

Why: every path and frozen setting is set in one place, so nothing can change between the dry run and the Test run except `RUN_ON`.

What the code does:
- Loads the libraries and Vincent's shared evaluation code (`load_calibration`, `run_evaluation` from `models/Transformer/evaluate_recon_model.py`).
- Sets the switch (`RUN_ON`, `UNLOCK_TEST`) and the frozen settings: `GRACE = 3`, `WARMUP = 23`, `MERGE_GAP = 2`, `SEEDS = [0, 1, 2]`, `SCORE = "topk_percentile"`.
- Sets the paths of all saved models and calibrations.
- Picks the data file, the attack windows file and the expected counts for `RUN_ON`. With `RUN_ON = "test"` and `UNLOCK_TEST = False` it stops with an error.
- Makes the output folder `evaluation/results/<RUN_ON>/`.

In [1]:
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"   # Windows: torch and numpy both ship OpenMP

import sys
import json
import subprocess
from datetime import datetime
from pathlib import Path
import numpy as np
import pandas as pd
import joblib

ROOT = Path("..")
sys.path.append(str(ROOT / "models" / "Transformer"))
from evaluate_recon_model import load_calibration, run_evaluation

# which data to score: "dataset2" (dry run) or "test" (only after freeze)
RUN_ON = "dataset2"
UNLOCK_TEST = False

# frozen settings
GRACE = 3
WARMUP = 23
MERGE_GAP = 2           # also stored in each frozen calibration; checked below
SEEDS = [0, 1, 2]
SCORE = "topk_percentile"

# paths
PROCESSED = ROOT / "data" / "processed"
RIDGE_MODEL = ROOT / "models" / "Ridge" / "ridge_model.joblib"
RIDGE_ERRORS = ROOT / "models" / "Ridge" / "ridge_errors.csv"
RIDGE_CALIB = ROOT / "models" / "Ridge" / "calibration" / "calibration.pt"
TR_WEIGHTS = ROOT / "models" / "Transformer" / "Notebook" / "Model Weights"
TR_MODEL = TR_WEIGHTS / "best_model.pt"
TR_CALIB = TR_WEIGHTS / "calibration" / "calibration.pt"
XGB_DIR = ROOT / "results" / "xgboost_iforest_ensemble" / "xgb"
IF_SCORES_DIR = ROOT / "results" / "xgboost_iforest_ensemble" / "iforest"
IF_MODELS_DIR = ROOT / "models" / "Ensemble" / "iforest_pca_models"
ENSEMBLE_RUNS = ROOT / "models" / "Ensemble" / "notebook_outputs" / "all_runs.csv"
RAW_TEST = ROOT / "data" / "BATADAL_test_dataset.csv"
RAW_DATASET2 = ROOT / "data" / "Training dataset 2.csv"

if RUN_ON == "test":
    if not UNLOCK_TEST:
        raise RuntimeError("Test set is locked. Set UNLOCK_TEST = True only after the team freezes.")
    DATA = PROCESSED / "test_eval.csv"
    WINDOWS_FILE = ROOT / "data" / "attack_windows_test.csv"
    EXPECTED_ROWS, EXPECTED_ATTACK_HOURS = 2089, 407
    EXPECTED_ATTACKS = list(range(8, 15))
elif RUN_ON == "dataset2":
    DATA = PROCESSED / "dataset2_eval.csv"
    WINDOWS_FILE = ROOT / "data" / "attack_windows_dataset2.csv"
    EXPECTED_ROWS, EXPECTED_ATTACK_HOURS = 4177, 492
    EXPECTED_ATTACKS = list(range(1, 8))
else:
    raise ValueError(f"RUN_ON must be 'dataset2' or 'test', not {RUN_ON!r}")

RESULTS = ROOT / "evaluation" / "results" / RUN_ON
OUT_DIR = RESULTS / "ridge"
OUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"RUN_ON = {RUN_ON} | data: {DATA.name} | results: {RESULTS}")

RUN_ON = dataset2 | data: dataset2_eval.csv | results: ..\evaluation\results\dataset2


## 2. Load the data

Why: before scoring, we check that the file is the one we expect, so a wrong or changed file stops the notebook early.

What the code does: loads the data file (`df`) and checks the number of rows, attack hours and attacks.
- Dataset 2: 4,177 rows, 492 attack hours, attacks 1 to 7.
- Test set: 2,089 rows, 407 attack hours, attacks 8 to 14.

In [2]:
df = pd.read_csv(DATA, parse_dates=["DATETIME"])

print(df.shape)
print("attack hours:", int(df["label"].sum()))
print("attacks:", sorted(df.loc[df["attack_id"] > 0, "attack_id"].unique()))

assert len(df) == EXPECTED_ROWS, "wrong number of rows"
assert int(df["label"].sum()) == EXPECTED_ATTACK_HOURS, "wrong number of attack hours"

(4177, 40)
attack hours: 492
attacks: [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7)]


### 2b. Attack windows

Why: the per-attack results (detected, delay, hours caught) need each attack's real start and end. The window file and the `attack_id` column must agree.

What the code does:
- Loads the attack windows (`windows`): `attack_id`, `real_start`, `real_end`, `real_hours`.
- Checks the attack list, that the rows are strictly hourly, and that each window covers exactly the hours with that `attack_id`.
- Checks that no attack hour falls in the first `WARMUP` = 23 hours, which are not scored.

In [3]:
windows = pd.read_csv(WINDOWS_FILE, parse_dates=["real_start", "real_end"])
print(windows.to_string(index=False))

assert sorted(df.loc[df["attack_id"] > 0, "attack_id"].unique()) == EXPECTED_ATTACKS, "wrong attacks"
assert sorted(windows["attack_id"]) == EXPECTED_ATTACKS, "window file has other attacks"
assert (df["DATETIME"].diff().dropna() == pd.Timedelta(hours=1)).all(), "rows are not strictly hourly"
for w in windows.itertuples():
    hours = df.loc[df["attack_id"] == w.attack_id, "DATETIME"]
    assert hours.min() == w.real_start and hours.max() == w.real_end and len(hours) == w.real_hours, w
assert (df["label"] == (df["attack_id"] > 0)).all(), "label and attack_id disagree"
assert df.loc[: WARMUP - 1, "label"].sum() == 0, "an attack hour falls in the warm-up hours"
print(f"first scored hour: {df.loc[WARMUP, 'DATETIME']} (row {WARMUP}); attack hours in the first {WARMUP} hours: 0")

 attack_id          real_start            real_end  real_hours
         1 2016-09-13 23:00:00 2016-09-16 00:00:00        50.0
         2 2016-09-26 11:00:00 2016-09-27 10:00:00        24.0
         3 2016-10-09 09:00:00 2016-10-11 20:00:00        60.0
         4 2016-10-29 19:00:00 2016-11-02 16:00:00        94.0
         5 2016-11-26 17:00:00 2016-11-29 04:00:00        60.0
         6 2016-12-06 07:00:00 2016-12-10 04:00:00        94.0
         7 2016-12-14 15:00:00 2016-12-19 04:00:00       110.0
first scored hour: 2016-07-04 23:00:00 (row 23); attack hours in the first 23 hours: 0


## 3. Ridge

### 3a. Ridge errors

Why: Ridge predicts each of the 28 continuous sensors from the other 35 features. A big error means the sensor does not fit the usual pattern. Ridge looks at one hour at a time, so it needs no warm-up.

What the code does:
- Loads the saved Ridge bundle. The 28 models are in `bundle["models"]`, one per sensor.
- Takes the sensor order from `ridge_errors.csv`, so the columns are in the same order as when the calibration was made.
- For each sensor, predicts it from the other features and keeps the squared error (`errors`), for every hour.

In [4]:
bundle = joblib.load(RIDGE_MODEL)
ridge_models = bundle["models"]
features = list(bundle["features"])
print(type(ridge_models), len(ridge_models), "models,", len(features), "features")

saved_cols = pd.read_csv(RIDGE_ERRORS, nrows=0).columns
sensor_order = [c for c in saved_cols if c not in ("DATETIME", "split")]
print(len(sensor_order), "sensors")

errors = pd.DataFrame(index=df.index)
for sensor in sensor_order:
    m = ridge_models[sensor]
    if hasattr(m, "feature_names_in_"):
        X = df[list(m.feature_names_in_)]
    else:
        X = df[[f for f in features if f != sensor]].to_numpy()
    pred = m.predict(X)
    errors[sensor] = (df[sensor] - pred) ** 2

print(errors.shape)

<class 'dict'> 28 models, 36 features
28 sensors
(4177, 28)


### 3b. Check against the saved Ridge errors

Why: this shows the saved Ridge models still give the same errors as when Ridge was trained. If a library version changed the models, we would see it here.

What the code does: on Dataset 2 only, compares our squared errors with the squared Dataset 2 errors saved in `ridge_errors.csv`, and stops if they differ by more than 1e-10. On the Test set there are no saved errors, so the check is skipped.

In [5]:
if RUN_ON == "dataset2":
    saved = pd.read_csv(RIDGE_ERRORS, parse_dates=["DATETIME"])
    saved = saved[saved["split"] == "dataset2"].reset_index(drop=True)
    assert (saved["DATETIME"].values == df["DATETIME"].values).all()
    diff = np.abs(errors[sensor_order].to_numpy() - saved[sensor_order].to_numpy() ** 2).max()
    print(f"max |difference| with ridge_errors.csv (squared): {diff:.1e}")
    assert diff < 1e-10, "Ridge errors differ from the saved ones"
else:
    print("Test run: there are no saved Ridge errors to compare with, so this check is skipped.")

max |difference| with ridge_errors.csv (squared): 4.7e-13


### 3c. Ridge results

Why: we score Ridge with the frozen calibration and Vincent's shared `run_evaluation`, the same code as the Transformer, so the two are counted the same way.

What the code does:
- Loads Ridge's frozen calibration (`models/Ridge/calibration/calibration.pt`) and checks its merge gap is 2.
- Runs `run_evaluation` on the hours from `WARMUP` onward, with `grace_hours = 3`. The thresholds come from the calibration file, so nothing is recalibrated.
- Saves the results (metrics, per-attack results, hourly scores and plots) to `evaluation/results/<RUN_ON>/ridge/`.

In [6]:
calibration, _ = load_calibration(RIDGE_CALIB)
assert calibration["merge_gap_hours"] == MERGE_GAP and calibration["top_k"] == 3

keep = df.index >= WARMUP
results = run_evaluation(
    calibration,
    errors.loc[keep, sensor_order].to_numpy(),
    df.loc[keep, "DATETIME"].to_numpy(),
    df.loc[keep, "label"].to_numpy(),
    df.loc[keep, "attack_id"].to_numpy(),
    sensor_order,
    OUT_DIR,
    grace_hours=GRACE,
)
ridge_metrics = results

####################################################################################################
Results on Dataset 2 (4,154 scored hours: 492 attack, 3,641 clean)
                                topk_percentile     mean                topk_ratio          
Attacks detected                7 of 7              7 of 7              7 of 7              
Missed attacks (Attack ID:)     none                none                none                
Mean delay (h, detected only)   7.7                 6.0                 7.7                 
False-alarm episodes per week   1.34                1.20                1.29                
Clean hours under alarm         0.91%               0.82%               0.96%               
Inside budget on Dataset 2      False               False               False               
Inside budget on validation     False               True                False               
Precision                       0.899               0.902               0.899           

## 4. Transformer

### 4a. Load the frozen model and its calibration

Why: the Transformer reads the last 12 hours and rebuilds them. Its errors on each sensor are turned into a score with its own frozen calibration.

What the code does:
- Loads `best_model.pt` and rebuilds the model with the settings stored in it (the same steps as `transformer_inference.ipynb`).
- Loads the frozen `calibration.pt` and checks it belongs to this model (same window and features), with merge gap 2 and the top 3 sensors.
- Checks the model is the 12-hour, seed 0 model.

In [7]:
import torch
from model import Transformer_Recon_Model
from batadal_dataset import build_dataloaders, DATASET2_FILE
from evaluate_recon_model import load_model, scaler_matches, sensor_errors

ok, ckpt = load_model(TR_MODEL)
assert ok, f"could not load {TR_MODEL}"
tr_model = Transformer_Recon_Model(
    in_dim=ckpt["in_dim"], embedding_dim=ckpt["embedding_dim"], hidden_dim=ckpt["hidden_dim"],
    num_heads=ckpt["num_heads"],
    num_latent_tokens=ckpt.get("num_latent_tokens", ckpt.get("num_encodings")),
    latent_dim=ckpt.get("latent_dim", ckpt.get("bottleneck_dim", 16)),
    num_window=ckpt["window"], num_enc_blocks=ckpt["num_enc_blocks"],
    num_dec_blocks=ckpt["num_dec_blocks"], dropout=ckpt["dropout"],
    activation_type=ckpt["activation_type"])
tr_model.load_state_dict(ckpt["model"])
tr_model = tr_model.eval()

tr_calibration, tr_saved = load_calibration(TR_CALIB)
window = ckpt["window"]
assert tr_saved["window"] == window and tr_saved["feature_names"] == ckpt["feature_names"]
assert window == 12 and ckpt.get("seed") == 0, "not the frozen 12-hour seed 0 model"
assert tr_calibration["merge_gap_hours"] == MERGE_GAP and tr_calibration["top_k"] == 3
print(f"window {window} h | seed {ckpt.get('seed')} | step {ckpt['curr_global_steps']:,} | "
      f"{SCORE} threshold {tr_calibration['thresholds'][SCORE]:.6f}")

window 12 h | seed 0 | step 25,000 | topk_percentile threshold 0.998448


### 4b. Transformer errors

Why: there are two ways in, and both must give the same numbers.
- **Dataset 2:** we score `dataset2_eval.csv` exactly as Vincent's code does.
- **Test set:** Vincent's loader only takes the RAW Test file (`build_dataloaders(test_path=..., unlock_test=True)`). It rounds, drops the 7 columns and scales with the saved scaler itself. Its Test hours have no labels, so we join our `label` and `attack_id` from `test_eval.csv` by `DATETIME`.

What the code does:
- `transformer_raw_errors()` runs Vincent's Test path on a raw file and returns the per-sensor errors and their hours.
- Dataset 2: scores `dataset2_eval.csv`. As a check of the Test path, it also runs `transformer_raw_errors()` on the raw Dataset 2 file (same format as the Test file) and checks the errors agree.
- Test set: runs `transformer_raw_errors()` on the raw Test file. This only runs when `UNLOCK_TEST` is `True`.
- Joins `label` and `attack_id` by `DATETIME` and checks the join: every scored hour is in our file, only the first 11 hours (window − 1) are not scored, and all attack hours are covered (407 on the Test set).

In [8]:
def transformer_raw_errors(raw_path):
    # Vincent's Test path: prepare a RAW file with the saved scaler, then score it.
    loaders, scaler, info = build_dataloaders(
        str(PROCESSED), None, test_path=str(raw_path), unlock_test=True,
        window=window, batch_size=128)
    assert info["feature_names"] == ckpt["feature_names"] and scaler_matches(scaler, ckpt)
    err, rows = sensor_errors(model=tr_model, dataloader=loaders["test"], device="cpu")
    return err, pd.to_datetime(info["test"]["datetime"][rows])


if RUN_ON == "dataset2":
    loaders, scaler, info = build_dataloaders(
        str(PROCESSED), str(PROCESSED / DATASET2_FILE), window=window, batch_size=128)
    assert info["feature_names"] == ckpt["feature_names"] and scaler_matches(scaler, ckpt)
    tr_errors, tr_rows = sensor_errors(model=tr_model, dataloader=loaders["dev"], device="cpu")
    tr_when = pd.to_datetime(info["dev"]["datetime"][tr_rows])

    # dry run of the Test path on the raw Dataset 2 file (same format as the raw Test file)
    raw_errors, raw_when = transformer_raw_errors(RAW_DATASET2)
    assert (raw_when.values == tr_when.values).all()
    print(f"Test path on raw Dataset 2 vs dataset2_eval.csv: max |error difference| "
          f"{np.abs(raw_errors - tr_errors).max():.1e}")
else:
    assert UNLOCK_TEST, "the Test set is locked"
    tr_errors, tr_when = transformer_raw_errors(RAW_TEST)

# join our labels by DATETIME
labels_by_time = df.set_index("DATETIME")[["label", "attack_id"]]
joined = labels_by_time.reindex(tr_when)
assert not joined.isna().any().any(), "some scored hours are not in our data file"
tr_labels = joined["label"].to_numpy().astype(int)
tr_attack_id = joined["attack_id"].to_numpy().astype(int)
assert len(tr_when) == len(df) - (window - 1), "unexpected number of scored hours"
assert (tr_when.values == df["DATETIME"].values[window - 1:]).all(), "scored hours are not the file's hours"
assert tr_labels.sum() == EXPECTED_ATTACK_HOURS, "the join lost attack hours"
print(f"{len(tr_when):,} scored hours of {len(df):,} (the first {window - 1} have no 12-hour history); "
      f"attack hours covered: {tr_labels.sum()} of {EXPECTED_ATTACK_HOURS}")

Test path on raw Dataset 2 vs dataset2_eval.csv: max |error difference| 0.0e+00
4,166 scored hours of 4,177 (the first 11 have no 12-hour history); attack hours covered: 492 of 492


### 4c. Transformer results

Why: the same evaluation as Ridge, with the Transformer's own frozen calibration.

What the code does: runs `run_evaluation` on the hours from `WARMUP` onward (so the same hours as every other model), with `grace_hours = 3`, and saves the results to `evaluation/results/<RUN_ON>/transformer/`.

In [9]:
TR_OUT = RESULTS / "transformer"
TR_OUT.mkdir(parents=True, exist_ok=True)

tr_keep = tr_when >= df.loc[WARMUP, "DATETIME"]
transformer_metrics = run_evaluation(
    tr_calibration,
    tr_errors[tr_keep],
    tr_when[tr_keep].to_numpy(),
    tr_labels[tr_keep],
    tr_attack_id[tr_keep],
    ckpt["feature_names"],
    str(TR_OUT),
    grace_hours=GRACE,
)

####################################################################################################
Results on Dataset 2 (4,154 scored hours: 492 attack, 3,641 clean)
                                topk_percentile     mean                topk_ratio          
Attacks detected                7 of 7              7 of 7              7 of 7              
Missed attacks (Attack ID:)     none                none                none                
Mean delay (h, detected only)   1.7                 1.6                 9.9                 
False-alarm episodes per week   1.15                1.29                1.66                
Clean hours under alarm         0.88%               1.10%               1.10%               
Inside budget on Dataset 2      False               False               False               
Inside budget on validation     True                False               False               
Precision                       0.906               0.893               0.839           

## 5. XGBoost, IF + PCA and the ensemble

### 5a. Features

Why: Suhas's XGBoost and IF + PCA use extra features built from the last hours: lags, differences and rolling means over up to 24 hours. Hour 23 is the first hour with 24 hours behind it. So we build the features on **all** hours, and score from hour 23 onward. If we cut the first 23 hours before building the features, we would lose 23 more hours.

What the code does:
- Builds the features with Suhas's `build_features()` (`models/XGBoost/xgb_data.py`), without time features, as in his frozen runs. It ignores `ATT_FLAG`, `label` and `attack_id`, so it works on the Test file too.
- Checks the columns are the same as the Training features, and keeps the hours from `WARMUP` onward (`feats`, `meta`).
- Builds the Calibration block features the same way (`cal_feats`). These are only used in the checks below.

In [10]:
sys.path.append(str(ROOT / "models" / "XGBoost"))
sys.path.append(str(ROOT / "evaluation"))
import xgboost as xgb
from xgb_data import build_features, WARMUP as XGB_WARMUP
import compare_models as CM

assert XGB_WARMUP == WARMUP
feature_cols = list(build_features(pd.read_csv(PROCESSED / "train_scaled.csv", nrows=50), time_features=False).columns)

feats_all = build_features(pd.read_csv(DATA), time_features=False)   # all hours, so hour 23 has its history
assert list(feats_all.columns) == feature_cols, "feature columns differ from Training"
scored = np.arange(len(df)) >= WARMUP
feats = feats_all[scored].reset_index(drop=True)
assert not feats.isna().any().any(), "missing values after the warm-up hours"
meta = df.loc[scored, ["DATETIME", "label", "attack_id"]].reset_index(drop=True)
meta["row"] = np.flatnonzero(scored)

cal_feats = build_features(pd.read_csv(PROCESSED / "cal_scaled.csv"), time_features=False)[WARMUP:].reset_index(drop=True)
print(f"{len(feature_cols)} features | {len(feats):,} scored hours | Calibration {len(cal_feats):,} hours")

387 features | 4,154 scored hours | Calibration 1,706 hours


### 5b. XGBoost scores

**How "in turn" (round-robin) works.** Suhas trained one XGBoost model per labeled attack, each without that attack (fold 1, 3, 4, 5 and 6). On data none of them trained on, his code gives each hour the score of one fold model, taken in turn by row position: row 0 gets fold 1, row 1 fold 3, row 2 fold 4, row 3 fold 5, row 4 fold 6, row 5 fold 1 again, and so on. So the score of an hour depends on its row position, not on the hour itself. On the Test set, row 0 is the first scored hour (hour 23). This is the same rule he used for the Validation and Calibration blocks.

**Dataset 2 is different.** Dataset 2 holds the attacks the folds were trained on. There, his saved scores give each attack window to the fold model that never saw that attack (attacks 2 and 7 to the final model). The hours outside the windows go in turn. We copy that here, so the dry run matches his saved scores. That choice uses the attack labels, so it can only be done on Dataset 2. On the Test set no fold model has seen any Test hour, so every hour goes in turn.

What the code does:
- Loads the 5 fold models and the final model per seed (`results/xgboost_iforest_ensemble/xgb/seed{k}/models/`), and checks the fold order against `thresholds.csv`.
- `xgb_in_turn()`: the round-robin rule above.
- Check (both runs): scores the Calibration block in turn and compares with his saved Calibration scores. This tests the exact code the Test set uses.
- Scores our data: in turn on the Test set; his Dataset 2 rule on Dataset 2, then compares with his saved Dataset 2 scores.

In [11]:
FOLDS = ["fold1", "fold3", "fold4", "fold5", "fold6"]


def load_xgb(seed):
    names = pd.read_csv(XGB_DIR / f"seed{seed}" / "models" / "thresholds.csv")["model"].tolist()
    assert [n for n in names if n != "final"] == FOLDS and "final" in names
    boosters = {}
    for name in names:
        b = xgb.Booster()
        b.load_model(str(XGB_DIR / f"seed{seed}" / "models" / f"{name}.json"))
        boosters[name] = b
    return boosters


def predict(booster, X):
    return booster.predict(xgb.DMatrix(np.asarray(X))).astype(np.float64)


def xgb_in_turn(boosters, X):
    # Row i is scored by FOLDS[i % 5]: the same rule as single() in train_xgboost.py.
    all_folds = np.stack([predict(boosters[n], X) for n in FOLDS])
    return all_folds[np.arange(len(X)) % len(FOLDS), np.arange(len(X))]


def xgb_dataset2(boosters, X, attack_id):
    # Suhas's saved Dataset 2 scores: each window by the fold that never saw it, the rest in turn.
    out = np.zeros(len(X))
    inside = attack_id > 0
    for a in np.unique(attack_id[inside]):
        sel = attack_id == a
        out[sel] = predict(boosters[f"fold{a}" if f"fold{a}" in FOLDS else "final"], X[sel])
    out[~inside] = xgb_in_turn(boosters, X[~inside])
    return out


def saved_scores(path, source, column):
    # round_trip: read each saved number back exactly (the default parser can be off by 1 bit)
    s = pd.read_csv(path, float_precision="round_trip")
    return s[s["source"] == source].sort_values("row").set_index("row")[column]


xgb_scores, xgb_cal_ref = {}, {}
for seed in SEEDS:
    boosters = load_xgb(seed)
    saved_path = XGB_DIR / f"seed{seed}" / "xgb_hourly_scores.csv"
    xgb_cal_ref[seed] = saved_scores(saved_path, "cal", "xgb_score").to_numpy()   # frozen Calibration reference

    cal_check = np.abs(xgb_in_turn(boosters, cal_feats.to_numpy()) - xgb_cal_ref[seed]).max()
    if RUN_ON == "dataset2":
        xgb_scores[seed] = xgb_dataset2(boosters, feats.to_numpy(), meta["attack_id"].to_numpy())
        d2_check = np.abs(xgb_scores[seed] - saved_scores(saved_path, "d2", "xgb_score").loc[meta["row"]].to_numpy()).max()
        print(f"seed {seed}: Calibration in turn vs saved {cal_check:.1e} | Dataset 2 vs saved {d2_check:.1e}")
        assert d2_check < 1e-6
    else:
        xgb_scores[seed] = xgb_in_turn(boosters, feats.to_numpy())
        print(f"seed {seed}: Calibration in turn vs saved {cal_check:.1e} | Test hours scored in turn: {len(feats):,}")
    assert cal_check < 1e-6, "saved XGBoost models do not reproduce the saved Calibration scores"

seed 0: Calibration in turn vs saved 2.9e-11 | Dataset 2 vs saved 4.7e-10


seed 1: Calibration in turn vs saved 2.9e-11 | Dataset 2 vs saved 7.5e-09


seed 2: Calibration in turn vs saved 1.2e-10 | Dataset 2 vs saved 6.0e-08


### 5c. IF + PCA scores

Why: the Isolation Forest finds hours that are easy to separate from normal Training hours. Suhas's version first scales the raw sensors and their 1-hour differences and reduces them to 20 PCA components. All three steps were fitted on Training only and saved.

What the code does:
- Loads the saved `{cols, scaler, pca, forest}` per seed (`models/Ensemble/iforest_pca_models/seed{k}.joblib`).
- Scores each hour as `-forest.score_samples(...)` (higher = more unusual), the same as `isolation_forest.py`.
- Check: scores the Calibration block and compares with his saved Calibration scores. On Dataset 2 it also compares our scores with his saved Dataset 2 scores.

In [12]:
def if_score(saved, X):
    cols = saved["cols"]
    return -saved["forest"].score_samples(saved["pca"].transform(saved["scaler"].transform(X[cols])))


if_scores, if_cal_ref = {}, {}
for seed in SEEDS:
    saved_if = joblib.load(IF_MODELS_DIR / f"seed{seed}.joblib")
    saved_path = IF_SCORES_DIR / f"seed{seed}" / "iforest_hourly_scores.csv"
    if_cal_ref[seed] = saved_scores(saved_path, "cal", "if_score").to_numpy()   # frozen Calibration reference

    cal_check = np.abs(if_score(saved_if, cal_feats) - if_cal_ref[seed]).max()
    if_scores[seed] = if_score(saved_if, feats)
    line = f"seed {seed}: Calibration vs saved {cal_check:.1e}"
    if RUN_ON == "dataset2":
        d2_check = np.abs(if_scores[seed] - saved_scores(saved_path, "d2", "if_score").loc[meta["row"]].to_numpy()).max()
        line += f" | Dataset 2 vs saved {d2_check:.1e}"
        assert d2_check < 1e-10
    print(line)
    assert cal_check < 1e-10, "saved IF + PCA models do not reproduce the saved Calibration scores"

seed 0: Calibration vs saved 5.6e-17 | Dataset 2 vs saved 5.6e-17


seed 1: Calibration vs saved 5.6e-17 | Dataset 2 vs saved 5.6e-17


seed 2: Calibration vs saved 5.6e-17 | Dataset 2 vs saved 5.6e-17


### 5d. Percentiles, frozen thresholds and alarms

Why: XGBoost and IF + PCA give scores on different scales. So each score is turned into a percentile: the share of that detector's Calibration scores that are at or below it (as `normalise()` in `models/Ensemble/ensemble.py`). The ensemble takes the larger of the two percentiles.

The thresholds are frozen. They are the percentile thresholds Suhas set on Calibration (`models/Ensemble/notebook_outputs/all_runs.csv`, his IF + PCA, percentile scale): one for XGBoost alone, one for IF + PCA alone, and one for the max ensemble, per seed. These are the thresholds the shared comparison uses. We do not use the per-fold thresholds in `thresholds.csv`, because those only score XGBoost model by model, not the percentile scores the ensemble needs.

We read the saved numbers exactly (`float_precision="round_trip"`). A few hours have a percentile exactly equal to the threshold. If the threshold were read even 1 bit too low, those hours would wrongly become alarms.

What the code does:
- Turns each score into a percentile against the frozen Calibration scores from 5b and 5c.
- Builds the ensemble score: the max of the two percentiles.
- Reads the 3 thresholds per seed and sets an alarm when the score is above its threshold.

In [13]:
def percentile(values, reference):
    return np.searchsorted(np.sort(reference), values, side="right") / len(reference)


# round_trip: read the frozen thresholds back exactly. Some hours have a percentile exactly equal
# to the threshold, so a threshold read 1 bit too low would turn them into alarms.
runs = pd.read_csv(ENSEMBLE_RUNS, float_precision="round_trip")
runs = runs[(runs["iforest"] == "own_pca") & (runs["norm"] == "percentile")]


def frozen_threshold(seed, detectors, fusion):
    row = runs[(runs["seed"] == seed) & (runs["detectors"] == detectors) & (runs["fusion"] == fusion)]
    assert len(row) == 1
    return float(row["threshold"].iloc[0])


DETECTORS = {"XGBoost": ("xgb", "single"), "IF + PCA": ("iforest", "single"), "Ensemble (max)": ("xgb+iforest", "max")}
scores, alarms, thresholds = {name: {} for name in DETECTORS}, {name: {} for name in DETECTORS}, {name: {} for name in DETECTORS}
for seed in SEEDS:
    p_xgb = percentile(xgb_scores[seed], xgb_cal_ref[seed])
    p_if = percentile(if_scores[seed], if_cal_ref[seed])
    for name, values in (("XGBoost", p_xgb), ("IF + PCA", p_if), ("Ensemble (max)", np.maximum(p_xgb, p_if))):
        thresholds[name][seed] = frozen_threshold(seed, *DETECTORS[name])
        scores[name][seed] = values
        alarms[name][seed] = (values > thresholds[name][seed]).astype(int)

print(pd.DataFrame(thresholds).rename_axis("seed").round(6).to_string())

       XGBoost  IF + PCA  Ensemble (max)
seed                                    
0     0.992380  0.992966        0.995897
1     0.991794  0.992966        0.996483
2     0.992380  0.992966        0.996483


### 5e. Results per seed and mean over seeds

Why: these three detectors give one score per hour, not one per sensor, so `run_evaluation` cannot take them. We use `evaluate()` from `evaluation/compare_models.py`, the code behind the shared comparison, with merge gap 2 and grace 3. We do **not** use `ensemble.evaluate()`: it needs `ATT_FLAG` values, which the Test file does not have.

What the code does:
- Runs `CM.evaluate()` for each detector and seed on the scored hours, with our labels and attack windows.
- Takes the mean over seeds 0, 1 and 2 with `CM.summarise()`, as the shared comparison does.
- Saves the hourly scores and alarms, the results per seed and the per-attack results to `evaluation/results/<RUN_ON>/xgboost/`, `/iforest_pca/` and `/ensemble/`.

In [14]:
hour_keys = [CM.hour_key(str(t)) for t in meta["DATETIME"]]
labels_by_hour = {CM.hour_key(str(t)): int(l) for t, l in zip(df["DATETIME"], df["label"])}
windows_cm = [{"id": int(w.attack_id), "start": CM.hour_key(str(w.real_start)),
               "end": CM.hour_key(str(w.real_end)), "hours": int(w.real_hours)} for w in windows.itertuples()]


def cm_evaluate(score, alarm):
    run = {h: (float(s), int(a)) for h, s, a in zip(hour_keys, score, alarm)}
    return CM.evaluate(run, hour_keys, labels_by_hour, windows_cm, MERGE_GAP, GRACE)


def hours_caught(alarm, attack_id):
    return {int(a): int(alarm[attack_id == a].sum()) for a in EXPECTED_ATTACKS}


FOLDER = {"XGBoost": "xgboost", "IF + PCA": "iforest_pca", "Ensemble (max)": "ensemble"}
seed_results, seed_attacks, mean_results = {}, [], {}
for name in DETECTORS:
    out = RESULTS / FOLDER[name]
    out.mkdir(parents=True, exist_ok=True)
    rows = []
    for seed in SEEDS:
        metrics, per_attack = cm_evaluate(scores[name][seed], alarms[name][seed])
        rows.append(metrics)
        caught = hours_caught(alarms[name][seed], meta["attack_id"].to_numpy())
        for a in per_attack:
            seed_attacks.append({"model": name, "seed": seed, "attack": a["attack"], "detected": a["detected"],
                                 "delay_hours": a["delay_hours"], "hours_caught": caught[a["attack"]], "attack_hours": a["hours"]})
        pd.DataFrame({"DATETIME": meta["DATETIME"], "label": meta["label"], "attack_id": meta["attack_id"],
                      "score": scores[name][seed], "alarm": alarms[name][seed]}).to_csv(out / f"hourly_scores_seed{seed}.csv", index=False)
    seed_results[name] = pd.DataFrame(rows, index=pd.Index(SEEDS, name="seed"))
    seed_results[name]["threshold"] = [thresholds[name][s] for s in SEEDS]
    seed_results[name].to_csv(out / "metrics_per_seed.csv")
    mean_results[name], _ = CM.summarise(rows)
    pd.DataFrame([a for a in seed_attacks if a["model"] == name]).to_csv(out / "per_attack.csv", index=False)

for name in DETECTORS:
    print(name)
    print(seed_results[name][["attacks_detected", "mean_delay_hours", "precision", "recall", "f1",
                              "roc_auc", "pr_auc", "episodes_per_week", "s_score"]].round(3).to_string())
    print()

XGBoost
      attacks_detected  mean_delay_hours  precision  recall     f1  roc_auc  pr_auc  episodes_per_week  s_score
seed                                                                                                           
0                    4              11.5      0.718   0.254  0.375    0.752   0.448              0.923    0.540
1                    5              29.6      0.729   0.262  0.386    0.741   0.445              0.784    0.547
2                    4              11.5      0.750   0.262  0.389    0.751   0.441              0.831    0.542

IF + PCA
      attacks_detected  mean_delay_hours  precision  recall     f1  roc_auc  pr_auc  episodes_per_week  s_score
seed                                                                                                           
0                    6            13.667      0.807   0.297  0.434    0.673   0.428              1.338    0.690
1                    6             9.000      0.791   0.301  0.436    0.673   0.431   

## 6. Summary of all models

Why: one table with every model, counted the same way, plus a per-attack table.

**We report `topk_percentile` only for Ridge and the Transformer**, because it is the score we froze. `run_evaluation` also prints `mean` and `topk_ratio`. On the Test set we do not switch to one of them, even if it looks better. Picking a score after seeing Test results would break the freeze.

What the code does:
- Overall table (`overall`): S, precision, recall, F1, ROC-AUC, PR-AUC, mean delay, false-alarm episodes per week, share of normal hours under alarm, attacks detected, and inside budget. For XGBoost, IF + PCA and the ensemble it is the mean over the 3 seeds, and "inside budget" says in how many seeds.
- Every row is counted by the same code, `CM.evaluate()` from `evaluation/compare_models.py`, as in the shared comparison. For Ridge and the Transformer it takes their `topk_percentile` scores and alarms from `run_evaluation`.
- Inside budget means at most 1 false-alarm episode per week **and** at most 1% of normal hours under alarm.
- Check: for Ridge and the Transformer, `CM.evaluate()` and `run_evaluation` must agree on everything that comes from the alarms (S, precision, recall, delay, episodes). Their ROC-AUC and PR-AUC can differ a little (about 1e-4), because the two files use slightly different formulas for the area. The full `run_evaluation` results stay in `ridge/` and `transformer/`.
- Per-attack table (`per_attack`): detected, delay from the real start, and hours caught out of the attack's hours, for every model, seed and attack.
- Saves `overall.csv`, `per_attack.csv` and `settings.json` to `evaluation/results/<RUN_ON>/`.

In [15]:
def from_run_evaluation(name, metrics, out_dir):
    k = SCORE + "_"
    hourly = pd.read_csv(Path(out_dir) / "hourly_scores.csv", float_precision="round_trip")
    assert [CM.hour_key(t) for t in hourly["datetime"]] == hour_keys
    cm, per_attack = cm_evaluate(hourly[f"{SCORE}_score"], hourly[f"{SCORE}_alarm"])
    # everything that comes from the alarms must agree with run_evaluation
    for a, b in (("s_score", "s_score"), ("precision", "precision"), ("recall", "recall"), ("f1", "f1"),
                 ("false_alarm_episodes_per_week", "episodes_per_week"), ("mean_delay_hours", "mean_delay_hours"),
                 ("attacks_detected", "attacks_detected")):
        assert abs(metrics[k + a] - cm[b]) < 1e-9, (name, a, metrics[k + a], cm[b])
    assert (metrics[k + "dev_in_budget"]) == (cm["episodes_per_week"] <= 1.0 and 1 - cm["tnr"] <= 0.01)
    print(f"{name}: run_evaluation and CM.evaluate agree on the alarms; "
          f"ROC-AUC {metrics[k + 'roc_auc']:.6f} vs {cm['roc_auc']:.6f}, PR-AUC {metrics[k + 'pr_auc']:.6f} vs {cm['pr_auc']:.6f}")
    caught = hours_caught(hourly[f"{SCORE}_alarm"].to_numpy(), hourly["attack_id"].to_numpy())
    attacks = [{"model": name, "seed": "", "attack": a["attack"], "detected": a["detected"], "delay_hours": a["delay_hours"],
                "hours_caught": caught[a["attack"]], "attack_hours": a["hours"]} for a in per_attack]
    return cm, attacks


def table_row(name, runs, m, inside):
    return {"model": name, "runs": runs, "s_score": m["s_score"], "precision": m["precision"],
            "recall": m["recall"], "f1": m["f1"], "roc_auc": m["roc_auc"], "pr_auc": m["pr_auc"],
            "mean_delay_hours": m["mean_delay_hours"], "episodes_per_week": m["episodes_per_week"],
            "normal_hours_alarmed": 1 - m["tnr"], "attacks_detected": m["attacks_detected"], "inside_budget": inside}


overall_rows, attack_rows = [], []
for name, metrics, out_dir in (("Ridge", ridge_metrics, OUT_DIR), ("Transformer (12 h, seed 0)", transformer_metrics, TR_OUT)):
    cm, attacks = from_run_evaluation(name, metrics, out_dir)
    inside = "yes" if cm["episodes_per_week"] <= 1.0 and 1 - cm["tnr"] <= 0.01 else "no"
    overall_rows.append(table_row(name, 1, cm, inside))
    attack_rows += attacks

for name in DETECTORS:
    per_seed = seed_results[name]
    inside = ((per_seed["episodes_per_week"] <= 1.0) & (1 - per_seed["tnr"] <= 0.01)).sum()
    overall_rows.append(table_row(name, len(SEEDS), mean_results[name], f"{inside} of {len(SEEDS)} seeds"))
attack_rows += seed_attacks

overall = pd.DataFrame(overall_rows)
per_attack = pd.DataFrame(attack_rows)
overall.to_csv(RESULTS / "overall.csv", index=False)
per_attack.to_csv(RESULTS / "per_attack.csv", index=False)

shown = overall.copy()
shown["normal_hours_alarmed"] = (100 * shown["normal_hours_alarmed"]).round(2).astype(str) + "%"
print(f"{RUN_ON}: {len(hour_keys):,} scored hours, merge gap {MERGE_GAP}, grace {GRACE}, score {SCORE}")
print(shown.round(3).to_string(index=False))

Ridge: run_evaluation and CM.evaluate agree on the alarms; ROC-AUC 0.864751 vs 0.864755, PR-AUC 0.734761 vs 0.734612
Transformer (12 h, seed 0): run_evaluation and CM.evaluate agree on the alarms; ROC-AUC 0.919427 vs 0.919422, PR-AUC 0.791807 vs 0.791655
dataset2: 4,154 scored hours, merge gap 2, grace 3, score topk_percentile
                     model  runs  s_score  precision  recall    f1  roc_auc  pr_auc  mean_delay_hours  episodes_per_week normal_hours_alarmed  attacks_detected inside_budget
                     Ridge     1    0.840      0.899   0.598 0.718    0.865   0.735             7.714              1.338                0.91%             7.000            no
Transformer (12 h, seed 0)     1    0.895      0.906   0.630 0.743    0.919   0.792             1.714              1.154                0.88%             7.000            no
                   XGBoost     3    0.543      0.732   0.259 0.383    0.748   0.445            17.533              0.846                1.28%        

### 6b. Per-attack table

What the code does: shows, for every attack, whether each model detected it, the delay in hours from the real start, and the hours caught out of the attack's hours. For the 3-seed models the three seeds are shown as 0 / 1 / 2.

In [16]:
def cell(group):
    parts = []
    for r in group.itertuples():
        delay = "miss" if not r.detected else f"{r.delay_hours:.0f} h"
        parts.append(f"{delay}, {r.hours_caught}/{r.attack_hours}")
    return " | ".join(parts)


table = per_attack.groupby(["model", "attack"], sort=False).apply(cell, include_groups=False).unstack("attack")
table.columns = [f"attack {a}" for a in table.columns]
with pd.option_context("display.max_colwidth", None, "display.width", 250):
    print("delay from the real start, hours caught / attack hours (seeds 0 | 1 | 2 for the 3-seed models)")
    print(table.to_string())

delay from the real start, hours caught / attack hours (seeds 0 | 1 | 2 for the 3-seed models)
                                                        attack 1                              attack 2                                 attack 3                                 attack 4                              attack 5                                 attack 6                                   attack 7
model                                                                                                                                                                                                                                                                                                             
Ridge                                                 0 h, 14/50                            0 h, 19/24                              39 h, 17/60                               0 h, 89/94                            0 h, 60/60                              10 h, 69/94                            

### 6c. Settings

What the code does: saves `settings.json` with the run (`RUN_ON`, time, git commit), the frozen settings and files, the hours scored and the library versions, so the results can be traced back.

In [17]:
import sklearn

try:
    commit = subprocess.run(["git", "rev-parse", "HEAD"], capture_output=True, text=True, cwd=ROOT).stdout.strip()
except OSError:
    commit = "unknown"

settings = {
    "run_on": RUN_ON, "unlock_test": UNLOCK_TEST, "run_at": datetime.now().isoformat(timespec="seconds"),
    "git_commit": commit, "data": str(DATA), "attack_windows": str(WINDOWS_FILE),
    "rows": len(df), "attack_hours": int(df["label"].sum()), "scored_hours": len(hour_keys),
    "first_scored_hour": str(meta["DATETIME"].iloc[0]),
    "warmup_hours": WARMUP, "merge_gap_hours": MERGE_GAP, "grace_hours": GRACE, "score": SCORE, "top_k": 3,
    "budget": {"max_alarm_fraction": 0.01, "max_episodes_per_week": 1.0}, "seeds": SEEDS,
    "frozen": {
        "ridge": {"model": str(RIDGE_MODEL), "calibration": str(RIDGE_CALIB),
                  "threshold": float(calibration["thresholds"][SCORE])},
        "transformer": {"model": str(TR_MODEL), "calibration": str(TR_CALIB), "window": window,
                        "seed": ckpt.get("seed"), "threshold": float(tr_calibration["thresholds"][SCORE])},
        "xgboost": {"models": str(XGB_DIR / "seed{k}" / "models"), "scoring": "fold models in turn by row position",
                    "calibration_reference": str(XGB_DIR / "seed{k}" / "xgb_hourly_scores.csv")},
        "iforest_pca": {"models": str(IF_MODELS_DIR / "seed{k}.joblib"),
                        "calibration_reference": str(IF_SCORES_DIR / "seed{k}" / "iforest_hourly_scores.csv")},
        "ensemble": {"fusion": "max of calibration percentiles", "thresholds_from": str(ENSEMBLE_RUNS)},
        "thresholds": {name: {str(s): t for s, t in thresholds[name].items()} for name in DETECTORS}},
    "versions": {"python": sys.version.split()[0], "numpy": np.__version__, "pandas": pd.__version__,
                 "scikit-learn": sklearn.__version__, "torch": torch.__version__, "xgboost": xgb.__version__},
}
(RESULTS / "settings.json").write_text(json.dumps(settings, indent=2))
print("saved", RESULTS / "overall.csv", RESULTS / "per_attack.csv", RESULTS / "settings.json")

saved

 ..\evaluation\results\dataset2\overall.csv ..\evaluation\results\dataset2\per_attack.csv ..\evaluation\results\dataset2\settings.json


## 7. Check against the shared comparison (dry run only)

Why: on Dataset 2 every model must give the same numbers as the shared comparison (`evaluation/results/comparison_gap2_grace3.csv`, 4,154 scored hours). If they match, the notebook scores each frozen model the same way as before, and we can trust it on the Test set.

What the code does: on Dataset 2, compares our overall table with the shared comparison, metric by metric, and stops if any number differs by more than 1e-4. On the Test set there is nothing to compare with, so it is skipped.

In [18]:
if RUN_ON == "dataset2":
    shared = pd.read_csv(ROOT / "evaluation" / "results" / "comparison_gap2_grace3.csv").set_index("model")
    names = {"Ridge": "Ridge", "Transformer (12 h, seed 0)": "Transformer AE", "XGBoost": "XGBoost",
             "IF + PCA": "IF + PCA (Suhas)", "Ensemble (max)": "Ensemble XGB + IF (max)"}
    cols = ["s_score", "attacks_detected", "mean_delay_hours", "episodes_per_week", "precision", "recall",
            "f1", "roc_auc", "pr_auc"]
    check = []
    for ours_name, shared_name in names.items():
        ours = overall.set_index("model").loc[ours_name]
        theirs = shared.loc[shared_name]
        assert int(theirs["scored_hours"]) == len(hour_keys)
        worst = max(abs(float(ours[c]) - float(theirs[c])) for c in cols)
        check.append({"model": ours_name, "S ours": ours["s_score"], "S shared": theirs["s_score"],
                      "attacks ours": ours["attacks_detected"], "attacks shared": theirs["attacks_detected"],
                      "delay ours": ours["mean_delay_hours"], "delay shared": theirs["mean_delay_hours"],
                      "FA/week ours": ours["episodes_per_week"], "FA/week shared": theirs["episodes_per_week"],
                      "largest difference": worst, "match": worst < 1e-4})
    check = pd.DataFrame(check)
    print(check.round(4).to_string(index=False))
    assert check["match"].all(), "the dry run does not match the shared comparison"
else:
    print("Test run: there is no shared comparison to check against, so this check is skipped.")

                     model  S ours  S shared  attacks ours  attacks shared  delay ours  delay shared  FA/week ours  FA/week shared  largest difference  match
                     Ridge  0.8399    0.8399        7.0000          7.0000      7.7143        7.7143        1.3381          1.3381                 0.0   True
Transformer (12 h, seed 0)  0.8946    0.8946        7.0000          7.0000      1.7143        1.7143        1.1535          1.1535                 0.0   True
                   XGBoost  0.5433    0.5433        4.3333          4.3333     17.5333       17.5333        0.8459          0.8459                 0.0   True
                  IF + PCA  0.6962    0.6962        6.0000          6.0000     12.1111       12.1111        1.2612          1.2612                 0.0   True
            Ensemble (max)  0.7462    0.7462        7.0000          7.0000     20.5714       20.5714        0.9228          0.9228                 0.0   True


## End of notebook

The full write-up is in `docs/test_scoring.md`. The code cell below is empty.

Saved in `evaluation/results/<RUN_ON>/`: `overall.csv`, `per_attack.csv`, `settings.json`, and one folder per model (`ridge/`, `transformer/`, `xgboost/`, `iforest_pca/`, `ensemble/`).

Reminder: the Test set is scored once only, with `RUN_ON = "test"` and `UNLOCK_TEST = True`, after the team freezes all choices and thresholds.